In [1]:
from pyspark.sql import SparkSession
spark = (
    SparkSession.builder
    .appName("OSIPTEL_Procesamiento_Limpieza")
    .master("yarn")
    .config("spark.hadoop.fs.defaultFS", "hdfs://localhost:9000")
    .getOrCreate()
)
print("Spark:", spark.version)
print("Master:", spark.sparkContext.master)

26/10/01 21:49:28 WARN Utils: Your hostname, debian resolves to a loopback address: 127.0.0.1; using 10.0.2.15 instead (on interface enp0s3)
26/10/01 21:49:28 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/01 21:49:28 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/10/01 21:49:29 WARN Client: Neither spark.yarn.jars nor spark.yarn.archive is set, falling back to uploading libraries under SPARK_HOME.


Spark: 3.5.1
Master: yarn


In [2]:
rutas_raw = {
    "internet_fijo_2_1":
        "hdfs://localhost:9000/datalake/osiptel/raw/csv/internet_fijo_2_1.csv",
    "fibra_optica_2_2":
        "hdfs://localhost:9000/datalake/osiptel/raw/csv/fibra_optica_2_2.csv",
    "internet_fijo_distrital_2_3":
        "hdfs://localhost:9000/datalake/osiptel/raw/csv/internet_fijo_distrital_2_3.csv",
    "telefonia_fija_4_1":
        "hdfs://localhost:9000/datalake/osiptel/raw/csv/telefonia_fija_4_1.csv"
}

In [4]:
df_21 = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(rutas_raw["internet_fijo_2_1"])
)
df_22 = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(rutas_raw["fibra_optica_2_2"])
)
df_23 = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(rutas_raw["internet_fijo_distrital_2_3"])
)
df_41 = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(rutas_raw["telefonia_fija_4_1"])
)

In [5]:
print("2.1 Internet Fijo:", df_21.count())
print("2.2 Fibra Óptica:", df_22.count())
print("2.3 Internet Fijo Distrital:", df_23.count())
print("4.1 Telefonía Fija:", df_41.count())

2.1 Internet Fijo: 141106
2.2 Fibra Óptica: 868
2.3 Internet Fijo Distrital: 157310
4.1 Telefonía Fija: 52674


In [6]:
print("=== 2.1 INTERNET FIJO ===")
df_21.printSchema()
print("\n=== 2.2 FIBRA ÓPTICA ===")
df_22.printSchema()
print("\n=== 2.3 INTERNET FIJO DISTRITAL ===")
df_23.printSchema()
print("\n=== 4.1 TELEFONÍA FIJA ===")
df_41.printSchema()

=== 2.1 INTERNET FIJO ===
root
 |-- Mes: date (nullable = true)
 |-- Empresa: string (nullable = true)
 |-- Tecnologia: string (nullable = true)
 |-- Segmento: string (nullable = true)
 |-- Departamento: string (nullable = true)
 |-- Rango Velocidad de Bajada (nominal): string (nullable = true)
 |-- Conexiones: integer (nullable = true)


=== 2.2 FIBRA ÓPTICA ===
root
 |-- Mes: date (nullable = true)
 |-- Empresa: string (nullable = true)
 |-- Región: string (nullable = true)
 |-- Conexiones: integer (nullable = true)


=== 2.3 INTERNET FIJO DISTRITAL ===
root
 |-- Periodo: date (nullable = true)
 |-- Empresa: string (nullable = true)
 |-- Departamento: string (nullable = true)
 |-- Provincia: string (nullable = true)
 |-- Distrito: string (nullable = true)
 |-- Tecnología: string (nullable = true)
 |-- Segmento: string (nullable = true)
 |-- Conexiones: integer (nullable = true)


=== 4.1 TELEFONÍA FIJA ===
root
 |-- Empresa: string (nullable = true)
 |-- Departamento: string (nullabl

In [9]:
import unicodedata
import re
def normalizar_nombre(nombre):
    nombre = unicodedata.normalize("NFKD", nombre)
    nombre = "".join(c for c in nombre if not unicodedata.combining(c))
    nombre = nombre.lower()
    nombre = re.sub(r"[^a-z0-9]+", "_", nombre)
    return nombre.strip("_")
def normalizar_columnas(df):
    for columna in df.columns:
        df = df.withColumnRenamed(columna, normalizar_nombre(columna))
    return df
df_21 = normalizar_columnas(df_21)
df_22 = normalizar_columnas(df_22)
df_23 = normalizar_columnas(df_23)
df_41 = normalizar_columnas(df_41)

print("2.1:", df_21.columns)
print("2.2:", df_22.columns)
print("2.3:", df_23.columns)
print("4.1:", df_41.columns)

2.1: ['mes', 'empresa', 'tecnologia', 'segmento', 'departamento', 'rango_velocidad_de_bajada_nominal', 'conexiones']
2.2: ['mes', 'empresa', 'region', 'conexiones']
2.3: ['periodo', 'empresa', 'departamento', 'provincia', 'distrito', 'tecnologia', 'segmento', 'conexiones']
4.1: ['empresa', 'departamento', 'medio_de_acceso', 'mes', 'lineas']


In [10]:
from pyspark.sql import functions as F
def resumen_nulos(df, nombre):
    print(f"\n=== {nombre} ===")
    df.select([
        F.sum(F.col(c).isNull().cast("int")).alias(c)
        for c in df.columns
    ]).show(truncate=False)
resumen_nulos(df_21, "2.1 Internet Fijo")
resumen_nulos(df_22, "2.2 Fibra Óptica")
resumen_nulos(df_23, "2.3 Internet Fijo Distrital")
resumen_nulos(df_41, "4.1 Telefonía Fija")


=== 2.1 Internet Fijo ===
+---+-------+----------+--------+------------+---------------------------------+----------+
|mes|empresa|tecnologia|segmento|departamento|rango_velocidad_de_bajada_nominal|conexiones|
+---+-------+----------+--------+------------+---------------------------------+----------+
|0  |91     |0         |0       |201         |13029                            |0         |
+---+-------+----------+--------+------------+---------------------------------+----------+


=== 2.2 Fibra Óptica ===
+---+-------+------+----------+
|mes|empresa|region|conexiones|
+---+-------+------+----------+
|0  |0      |0     |0         |
+---+-------+------+----------+


=== 2.3 Internet Fijo Distrital ===
+-------+-------+------------+---------+--------+----------+--------+----------+
|periodo|empresa|departamento|provincia|distrito|tecnologia|segmento|conexiones|
+-------+-------+------------+---------+--------+----------+--------+----------+
|0      |0      |0           |0        |0    

In [12]:
def contar_duplicados(df, nombre):
    total = df.count()
    unicos = df.dropDuplicates().count()
    duplicados = total - unicos
    print(
        f"{nombre}: total={total}, "
        f"únicos={unicos}, "
        f"duplicados={duplicados}"
    )
contar_duplicados(df_21, "2.1")
contar_duplicados(df_22, "2.2")
contar_duplicados(df_23, "2.3")
contar_duplicados(df_41, "4.1")

2.1: total=141106, únicos=138620, duplicados=2486
2.2: total=868, únicos=868, duplicados=0
2.3: total=157310, únicos=157310, duplicados=0
4.1: total=52674, únicos=52653, duplicados=21


In [14]:
from pyspark.sql import functions as F
df_21_limpio = (
    df_21
    .withColumn("mes", F.to_date("mes"))
    .withColumn("conexiones", F.col("conexiones").cast("long"))
)
df_22_limpio = (
    df_22
    .withColumn("mes", F.to_date("mes"))
    .withColumn("conexiones", F.col("conexiones").cast("long"))
)
df_23_limpio = (
    df_23
    .withColumn("periodo", F.to_date("periodo"))
    .withColumn("conexiones", F.col("conexiones").cast("long"))
)
df_41_limpio = (
    df_41
    .withColumn("mes", F.to_date("mes"))
    .withColumn("lineas", F.col("lineas").cast("long"))
)

In [15]:
df_21_limpio = df_21_limpio.fillna({
    "empresa": "NO INFORMADO",
    "departamento": "NO INFORMADO",
    "rango_velocidad_de_bajada_nominal": "NO INFORMADO"
})
df_41_limpio = df_41_limpio.fillna({
    "departamento": "NO INFORMADO",
    "medio_de_acceso": "NO INFORMADO"
})

In [17]:
df_21_limpio = df_21_limpio.dropDuplicates()
df_22_limpio = df_22_limpio.dropDuplicates()
df_23_limpio = df_23_limpio.dropDuplicates()
df_41_limpio = df_41_limpio.dropDuplicates()

print("2.1 limpio:", df_21_limpio.count())
print("2.2 limpio:", df_22_limpio.count())
print("2.3 limpio:", df_23_limpio.count())
print("4.1 limpio:", df_41_limpio.count())

2.1 limpio: 138620
2.2 limpio: 868
2.3 limpio: 157310
4.1 limpio: 52653


In [19]:
resumen_nulos(df_21_limpio, "2.1 LIMPIO")
resumen_nulos(df_22_limpio, "2.2 LIMPIO")
resumen_nulos(df_23_limpio, "2.3 LIMPIO")
resumen_nulos(df_41_limpio, "4.1 LIMPIO")

contar_duplicados(df_21_limpio, "2.1 LIMPIO")
contar_duplicados(df_22_limpio, "2.2 LIMPIO")
contar_duplicados(df_23_limpio, "2.3 LIMPIO")
contar_duplicados(df_41_limpio, "4.1 LIMPIO")


=== 2.1 LIMPIO ===
+---+-------+----------+--------+------------+---------------------------------+----------+
|mes|empresa|tecnologia|segmento|departamento|rango_velocidad_de_bajada_nominal|conexiones|
+---+-------+----------+--------+------------+---------------------------------+----------+
|0  |0      |0         |0       |0           |0                                |0         |
+---+-------+----------+--------+------------+---------------------------------+----------+


=== 2.2 LIMPIO ===
+---+-------+------+----------+
|mes|empresa|region|conexiones|
+---+-------+------+----------+
|0  |0      |0     |0         |
+---+-------+------+----------+


=== 2.3 LIMPIO ===
+-------+-------+------------+---------+--------+----------+--------+----------+
|periodo|empresa|departamento|provincia|distrito|tecnologia|segmento|conexiones|
+-------+-------+------------+---------+--------+----------+--------+----------+
|0      |0      |0           |0        |0       |0         |0       |0     

In [21]:
ruta_silver = "hdfs://localhost:9000/datalake/osiptel/silver"
df_21_limpio.write \
    .mode("overwrite") \
    .parquet(f"{ruta_silver}/internet_fijo_2_1")
df_22_limpio.write \
    .mode("overwrite") \
    .parquet(f"{ruta_silver}/fibra_optica_2_2")
df_23_limpio.write \
    .mode("overwrite") \
    .parquet(f"{ruta_silver}/internet_fijo_distrital_2_3")
df_41_limpio.write \
    .mode("overwrite") \
    .parquet(f"{ruta_silver}/telefonia_fija_4_1")
print("Datos limpios almacenados correctamente en Silver")

Datos limpios almacenados correctamente en Silver


In [22]:
rutas_silver = {
    "internet_fijo_2_1":
        "hdfs://localhost:9000/datalake/osiptel/silver/internet_fijo_2_1",
    "fibra_optica_2_2":
        "hdfs://localhost:9000/datalake/osiptel/silver/fibra_optica_2_2",
    "internet_fijo_distrital_2_3":
        "hdfs://localhost:9000/datalake/osiptel/silver/internet_fijo_distrital_2_3",
    "telefonia_fija_4_1":
        "hdfs://localhost:9000/datalake/osiptel/silver/telefonia_fija_4_1"
}

In [24]:
silver_21 = spark.read.parquet(rutas_silver["internet_fijo_2_1"])
silver_22 = spark.read.parquet(rutas_silver["fibra_optica_2_2"])
silver_23 = spark.read.parquet(rutas_silver["internet_fijo_distrital_2_3"])
silver_41 = spark.read.parquet(rutas_silver["telefonia_fija_4_1"])

print("2.1 Silver:", silver_21.count())
print("2.2 Silver:", silver_22.count())
print("2.3 Silver:", silver_23.count())
print("4.1 Silver:", silver_41.count())

2.1 Silver: 138620
2.2 Silver: 868
2.3 Silver: 157310
4.1 Silver: 52653


In [25]:
silver_21.printSchema()

root
 |-- mes: date (nullable = true)
 |-- empresa: string (nullable = true)
 |-- tecnologia: string (nullable = true)
 |-- segmento: string (nullable = true)
 |-- departamento: string (nullable = true)
 |-- rango_velocidad_de_bajada_nominal: string (nullable = true)
 |-- conexiones: long (nullable = true)



In [26]:
silver_21.show(5, truncate=False)

+----------+--------------------------+--------------------+-----------+--------------+---------------------------------+----------+
|mes       |empresa                   |tecnologia          |segmento   |departamento  |rango_velocidad_de_bajada_nominal|conexiones|
+----------+--------------------------+--------------------+-----------+--------------+---------------------------------+----------+
|2002-12-01|NO INFORMADO              |2) xDSL             |Comercial  |NO INFORMADO  |NO INFORMADO                     |169       |
|2013-03-01|AMERICATEL PERU S.A.      |1) Dial-Up Fijo     |Residencial|LIMA Y CALLAO |NO INFORMADO                     |154       |
|2013-03-01|ENTEL PERU S.A.           |4) Wimax            |Residencial|LIMA Y CALLAO |NO INFORMADO                     |204       |
|2013-06-01|GILAT TO HOME PERU S.A.   |7) Otras Tecnologías|Comercial  |OTRAS REGIONES|NO INFORMADO                     |99        |
|2013-06-01|TELEFONICA DEL PERU S.A.A.|2) xDSL             |Comercial